# 03 Baseline Modeling, Threshold Tuning, And Error Analysis

Purpose: compare baseline models, tune the decision threshold using business cost, evaluate once on test, and interpret model mistakes.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report, brier_score_loss
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path(r"C:\Users\muham\OneDrive\Desktop\ML Journey\Project\kkbox-churn")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
modeling_data = pd.read_csv(PROCESSED_DIR / "kkbox_modeling_data_v1.csv")

print("Shape:", modeling_data.shape)
print(modeling_data["is_churn"].value_counts(normalize=True).round(4))

## Split Data

Validation is used for model selection and threshold tuning. Test is untouched until final evaluation.

In [ ]:
X = modeling_data.drop(columns=["is_churn", "msno"])
y = modeling_data["is_churn"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, y_train.mean().round(4))
print("Valid:", X_valid.shape, y_valid.mean().round(4))
print("Test:", X_test.shape, y_test.mean().round(4))

## Preprocessing Pipeline

The pipeline keeps cleaning and modeling together. It prevents inconsistent preprocessing between train, validation, test, and future scoring data.

In [ ]:
categorical_features = ["gender", "city", "registered_via", "latest_payment_method_id", "latest_is_auto_renew", "latest_is_cancel"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

numeric_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_preprocessor, numeric_features), ("categorical", categorical_preprocessor, categorical_features)])

## Cost-Sensitive Threshold Function

In [ ]:
def evaluate_thresholds(y_true, y_proba, fp_cost=1, fn_cost=5):
    rows = []
    for threshold in np.linspace(0.01, 0.99, 99):
        y_pred = (y_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        total_cost = fp * fp_cost + fn * fn_cost
        rows.append({
            "threshold": threshold, "tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "cost_per_customer": total_cost / len(y_true),
        })
    return pd.DataFrame(rows)

## Model Comparison

Start simple, then add complexity. The selected model must improve the business decision, not just sound advanced.

In [ ]:
models = {
    "Logistic Regression": Pipeline([("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, random_state=42))]),
    "Balanced Logistic Regression": Pipeline([("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "HistGradientBoosting": Pipeline([("preprocessor", preprocessor), ("model", HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_leaf_nodes=31, random_state=42))]),
    "hgb_regularized": Pipeline([("preprocessor", preprocessor), ("model", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.1, random_state=42))]),
}

comparison_rows = []
trained_models = {}
for name, model in models.items():
    print("Training", name)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_valid)[:, 1]
    threshold_results = evaluate_thresholds(y_valid, proba, fp_cost=1, fn_cost=5)
    best = threshold_results.loc[threshold_results["cost_per_customer"].idxmin()]
    comparison_rows.append({
        "model": name, "threshold": best["threshold"], "precision": best["precision"], "recall": best["recall"],
        "f1": best["f1"], "roc_auc": roc_auc_score(y_valid, proba), "pr_auc": average_precision_score(y_valid, proba),
        "tn": best["tn"], "fp": best["fp"], "fn": best["fn"], "tp": best["tp"], "cost_per_customer": best["cost_per_customer"],
    })
    trained_models[name] = model

model_comparison = pd.DataFrame(comparison_rows).sort_values("cost_per_customer")
print(model_comparison.round(4).to_string(index=False))
model_comparison.to_csv(PROCESSED_DIR / "validation_model_comparison_v1.csv", index=False)

### Interpretation

HistGradientBoosting is the strongest model family here. The regularized HGB model is selected because it has the lowest validation cost and strong PR-AUC.

## Final Test Evaluation

In [ ]:
selected_model_name = "hgb_regularized"
selected_model = trained_models[selected_model_name]
selected_threshold = 0.13

test_pred_proba = selected_model.predict_proba(X_test)[:, 1]
test_pred = (test_pred_proba >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
print("Precision:", round(precision_score(y_test, test_pred), 4))
print("Recall:", round(recall_score(y_test, test_pred), 4))
print("F1:", round(f1_score(y_test, test_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_pred_proba), 4))
print("PR-AUC:", round(average_precision_score(y_test, test_pred_proba), 4))
print("Brier:", round(brier_score_loss(y_test, test_pred_proba), 4))
print(confusion_matrix(y_test, test_pred))

fp_cost, fn_cost = 1, 5
cost_per_customer = (fp * fp_cost + fn * fn_cost) / len(y_test)
print("Cost per customer:", round(cost_per_customer, 4))

### Interpretation

This is the final v1 test result. Do not change the threshold after seeing this unless starting a new model version.

## Error Analysis And Risk Bands

In [ ]:
test_analysis = X_test.copy()
test_analysis["actual"] = y_test.values
test_analysis["predicted_probability"] = test_pred_proba
test_analysis["predicted"] = test_pred

conditions = [
    (test_analysis["actual"] == 1) & (test_analysis["predicted"] == 1),
    (test_analysis["actual"] == 0) & (test_analysis["predicted"] == 0),
    (test_analysis["actual"] == 0) & (test_analysis["predicted"] == 1),
    (test_analysis["actual"] == 1) & (test_analysis["predicted"] == 0),
]
labels = ["true_positive", "true_negative", "false_positive", "false_negative"]
test_analysis["prediction_group"] = np.select(conditions, labels)
print(test_analysis["prediction_group"].value_counts())

test_analysis["risk_band"] = pd.cut(
    test_analysis["predicted_probability"],
    bins=[0, 0.05, 0.13, 0.30, 0.60, 1.0],
    labels=["0-5%", "5-13%", "13-30%", "30-60%", "60-100%"],
    include_lowest=True,
)
risk_band_summary = test_analysis.groupby("risk_band", observed=False).agg(
    users=("actual", "size"),
    churners=("actual", "sum"),
    churn_rate=("actual", "mean"),
    targeted_rate=("predicted", "mean"),
)
print(risk_band_summary.round(4).to_string())

## Permutation Importance

Permutation importance shows what the model relies on. It also helps detect suspicious shortcut features.

In [ ]:
X_valid_sample = X_valid.sample(n=30000, random_state=42)
y_valid_sample = y_valid.loc[X_valid_sample.index]

perm_importance = permutation_importance(
    selected_model, X_valid_sample, y_valid_sample,
    scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1,
)
permutation_importance_df = pd.DataFrame({
    "feature": X_valid_sample.columns,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print(permutation_importance_df.head(20).round(5).to_string(index=False))
permutation_importance_df.to_csv(PROCESSED_DIR / "selected_hgb_permutation_importance_v1.csv", index=False)

### Interpretation

The top features are latest transaction timing, latest cancellation, latest auto-renew, expiration timing, and payment method. Timing-feature dominance should be documented as a v1 limitation and inspected more deeply in v2.